In [1]:
# ==========================================
# CELL 1 — INSTALL & RESTART
# Run this cell alone, then restart kernel
# ==========================================
!pip install -q -U transformers trl bitsandbytes accelerate peft datasets openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 r

In [2]:
# ==========================================
# CELL 2 — FULL PIPELINE
# ==========================================

import torch
import re
import gc
import json
import os
import pandas as pd
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

# ==========================================
# CONFIGURATION
# ==========================================
model_id         = "Qwen/Qwen3-0.6B"
dataset_id       = "openai/gsm8k"
SFT_ADAPTER_PATH = "/kaggle/input/datasets/muskaanadwani/sft-trial-1-weights"

os.makedirs("/kaggle/working", exist_ok=True)

# ==========================================
# TEST PROMPTS — updated to match GSM8K style
# ==========================================
eval_data = [
    {"id":1,  "prompt":"A store sells apples for $0.75 each and oranges for $1.25 each. If Sarah buys 4 apples and 3 oranges, how much does she spend in total?"},
    {"id":2,  "prompt":"A train travels at 60 mph for 2.5 hours, then at 80 mph for 1.5 hours. What is the total distance traveled?"},
    {"id":3,  "prompt":"A class has 32 students. 3/8 of them play football and 1/4 of them play basketball. How many students play neither sport?"},
    {"id":4,  "prompt":"John earns $15 per hour and works 8 hours a day, 5 days a week. He saves 20% of his weekly earnings. How much does he save per week?"},
    {"id":5,  "prompt":"A rectangle has a perimeter of 54 cm. If its length is twice its width, what is the area of the rectangle?"},
    {"id":6,  "prompt":"A tank is 1/3 full of water. After adding 24 liters, it becomes 3/4 full. What is the total capacity of the tank?"},
    {"id":7,  "prompt":"A shopkeeper bought 50 items for $200 and sold them all for $270. What is the profit percentage?"},
    {"id":8,  "prompt":"Two friends split a restaurant bill. The bill was $84 before a 15% tip. They also have a coupon for $10 off the total after tip. How much does each person pay?"},
    {"id":9,  "prompt":"A car depreciates in value by 15% each year. If it was worth $20,000 when new, what is it worth after 2 years?"},
    {"id":10, "prompt":"A pipe can fill a tank in 6 hours and another pipe can fill the same tank in 4 hours. If both pipes are opened together, how long will it take to fill the tank?"},
]
TEST_PROMPTS = [d["prompt"] for d in eval_data]
print("✅ Test prompts ready.")

# ==========================================
# QUANTIZATION CONFIG
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

config = AutoConfig.from_pretrained(model_id)
config.tie_word_embeddings = False

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# REWARD FUNCTIONS
# ==========================================
def format_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        has_open  = "<think>" in completion
        has_close = "</think>" in completion
        if has_open and has_close:
            think_content = completion.split("<think>")[1].split("</think>")[0].strip()
            rewards.append(1.0 if len(think_content) >= 20 else -1.0)
        else:
            rewards.append(0.0)
    return rewards

def accuracy_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    rewards = []
    for i, completion in enumerate(completions):
        true_answer = answer[i].split("####")[-1].strip()
        numbers = re.findall(r'-?\d+', completion)
        if numbers and numbers[-1] == true_answer:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

# ==========================================
# DATASET
# ==========================================
dataset       = load_dataset(dataset_id, "main")
train_dataset = dataset["train"].select(range(200))
if "question" in train_dataset.column_names:
    train_dataset = train_dataset.rename_column("question", "prompt")
print("✅ Dataset ready.")

# ==========================================
# GENERATE & SAVE ANSWERS FUNCTION
# ==========================================
def generate_and_save_answers(model, tokenizer, trial_name):
    print(f"\n📝 Generating answers for {trial_name}...")
    model.eval()
    results = []

    for i, item in enumerate(eval_data):
        messages  = [{"role": "user", "content": item["prompt"]}]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id
            )

        new_ids  = output_ids[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

        results.append({
            "id":             item["id"],
            "prompt":         item["prompt"],
            "model_response": response
        })
        print(f"  Q{i+1}/10 done.")

    out_path = f"/kaggle/working/{trial_name}_answers.csv"
    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"✅ Saved to {out_path}")
    model.train()

# ==========================================
# TRIAL CONFIGURATIONS
# ==========================================
TRIALS = [
    {"name": "grpo_trial_1", "lr": 1e-5,  "beta": 0.04, "num_gen": 4},  # baseline
    {"name": "grpo_trial_2", "lr": 5e-6,  "beta": 0.04, "num_gen": 4},  # lower lr
    {"name": "grpo_trial_3", "lr": 3e-5,  "beta": 0.04, "num_gen": 4},  # higher lr
    {"name": "grpo_trial_4", "lr": 1e-5,  "beta": 0.10, "num_gen": 4},  # high kl penalty
    {"name": "grpo_trial_5", "lr": 1e-5,  "beta": 0.01, "num_gen": 8},  # low kl + bigger group
]

# ==========================================
# RESUME GUARD
# ==========================================
completed = [
    f.replace("_answers.csv", "")
    for f in os.listdir("/kaggle/working")
    if f.endswith("_answers.csv")
]
print(f"Already completed: {completed if completed else 'none'}")

# ==========================================
# TRIAL LOOP
# ==========================================
for trial in TRIALS:
    TRIAL_NAME    = trial["name"]
    LEARNING_RATE = trial["lr"]
    BETA          = trial["beta"]
    NUM_GEN       = trial["num_gen"]

    if TRIAL_NAME in completed:
        print(f"⏭️  Skipping {TRIAL_NAME} — already done.")
        continue

    print(f"\n{'='*50}")
    print(f"🔬 {TRIAL_NAME} | LR={LEARNING_RATE} | Beta={BETA} | Groups={NUM_GEN}")
    print(f"{'='*50}")

    gc.collect()
    torch.cuda.empty_cache()

    print("🔁 Reloading SFT model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        config=config,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model = PeftModel.from_pretrained(
        base_model,
        SFT_ADAPTER_PATH,
        is_trainable=True
    )
    model = model.to(torch.float32)
    print("✅ Model loaded.")

    training_args = GRPOConfig(
        output_dir=f"/kaggle/working/{TRIAL_NAME}",
        learning_rate=LEARNING_RATE,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        num_generations=NUM_GEN,
        generation_batch_size=NUM_GEN,
        beta=BETA,
        max_completion_length=512,
        fp16=False,
        bf16=False,
        save_steps=50,
        logging_steps=10,
        report_to="none"
    )

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=[format_reward_func, accuracy_reward_func],
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
        peft_config=None
    )

    print(f"🚀 Training {TRIAL_NAME}...")
    trainer.train()
    trainer.save_model(training_args.output_dir)
    print(f"💾 Saved to /kaggle/working/{TRIAL_NAME}")

    generate_and_save_answers(
        model=model,
        tokenizer=tokenizer,
        trial_name=TRIAL_NAME
    )

    del trainer, base_model, model
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# DONE
# ==========================================
print("\n" + "="*50)
print("🏁 ALL TRIALS COMPLETE")
print("="*50)
print("Files saved for Member 1:")
for f in sorted(os.listdir("/kaggle/working")):
    if f.endswith("_answers.csv"):
        print(f"  📄 /kaggle/working/{f}")
print("\nSend all *_answers.csv files to Member 1 for evaluation.")

✅ Test prompts ready.


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

✅ Model loaded.


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Training grpo_trial_5...


Step,Training Loss
10,0.000088
20,0.000008
30,0.000146
40,0.000257
50,0.000009
60,0.000125
70,-0.017611
80,0.017830
90,0.000082
100,0.000039


💾 Saved to /kaggle/working/grpo_trial_5

📝 Generating answers for grpo_trial_5...
  Q1/10 done.
  Q2/10 done.
  Q3/10 done.
  Q4/10 done.
  Q5/10 done.
  Q6/10 done.
  Q7/10 done.
  Q8/10 done.
  Q9/10 done.
  Q10/10 done.
✅ Saved to /kaggle/working/grpo_trial_5_answers.csv

🏁 ALL TRIALS COMPLETE
Files saved for Member 1:
  📄 /kaggle/working/grpo_trial_1_answers.csv
  📄 /kaggle/working/grpo_trial_2_answers.csv
  📄 /kaggle/working/grpo_trial_3_answers.csv
  📄 /kaggle/working/grpo_trial_4_answers.csv
  📄 /kaggle/working/grpo_trial_5_answers.csv

Send all *_answers.csv files to Member 1 for evaluation.
